# learned

> The review queue for `<cfg>/learned`: what Dhrona proposed waits on a branch until a person accepts or rejects it.

In [ ]:
#| default_exp learned

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq, test_fail

Dhrona's nightly batch never changes the files Ramabana briefs from. It commits each candidate it keeps on its own `pending/<id>` branch of the learned folder's git repository. The briefing and skill discovery read only the folder's files, which are its HEAD, so a pending candidate has no effect until a person accepts it. `--no-learned` leaves out the folder whatever it holds.

`pending` lists what waits, `diff` shows one candidate's change, `accept` moves it onto HEAD, and `reject` drops it. Each of `diff`, `accept` and `reject` returns a dict with `ok` and `id`, and `error` when `ok` is False, so a host such as Leela can show the result as it is. They take the lock a Dhrona batch holds, and refuse while a batch runs. `accept` also refuses a candidate that would take `AGENTS.md` past `LEARNED_MAX`, the 2,000 characters Dhrona allows one candidate, since accepted rounds add up.

`compact` asks Dhrona for a shorter `AGENTS.md` when accepted lessons crowd it. Dhrona's editor model merges and shortens the paragraphs Dhrona proposed, keeps every paragraph a person wrote byte for byte, and commits the result as one more pending candidate, which `diff`, `accept` and `reject` act on like any other. `compact` returns `{ok, id, before, after}`, where `before` and `after` are the file's length in characters, or `{ok: False, error}`. It needs the `dhrona` extra, and says so when Dhrona is missing. It runs one model turn, so it takes as long as that turn.

In [ ]:
#| export
import fcntl, re, subprocess, sys, tempfile
from contextlib import contextmanager
from pathlib import Path
from fastcore.script import call_parse

LEARNED = 'learned'   #: the folder under the config dir where Dhrona keeps what it learned
PENDING = 'pending/'  #: the branch prefix a candidate waits under until a person accepts it
LEARNED_MAX = 2000    #: characters of `AGENTS.md` Dhrona lets a candidate carry; `accept` keeps HEAD within it too
_ID = re.compile(r'\d{8}-\d{6}-[0-9a-f]{6}')
_AS = ('-c', 'user.name=Ramabana', '-c', 'user.email=ramabana@localhost')

def learned_folder(cfg='~/.config/ramabana'):
    "`<cfg>/learned`, whether or not a session runs with it."
    return Path(cfg).expanduser()/LEARNED

def _git(d, *a, check=True): return subprocess.run(['git', '-C', str(d), *a], capture_output=True, text=True, check=check)

@contextmanager
def _locked(folder):
    "Hold the lock a Dhrona batch takes on `folder`, beside it; yields False while a batch holds it."
    folder = Path(folder)
    folder.parent.mkdir(parents=True, exist_ok=True)
    with open(folder.parent/f'.{folder.name}.lock', 'w') as f:
        try: fcntl.flock(f, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except OSError: yield False; return
        yield True

def pending(folder):
    "Each candidate waiting for review, oldest first, as `{id, at, subject, files}`; `at` is epoch seconds."
    folder = Path(folder)
    if not (folder/'.git').exists(): return []
    out = _git(folder, 'for-each-ref', '--sort=committerdate', '--format=%(refname:short)%00%(committerdate:unix)%00%(subject)',
               f'refs/heads/{PENDING}').stdout
    rows = []
    for l in out.splitlines():
        ref, at, subj = l.split('\0')
        rows.append(dict(id=ref.removeprefix(PENDING), at=int(at), subject=subj,
                         files=_git(folder, 'diff', '--name-only', f'{ref}^', ref).stdout.split()))
    return rows

def _problem(folder, id):
    "Why `id` cannot be acted on in `folder`, or None."
    if not _ID.fullmatch(str(id or '')): return f'{id!r} is not a pending id; `pending` lists them'
    if not (Path(folder)/'.git').exists(): return f'{folder} holds nothing Dhrona proposed'
    if _git(folder, 'rev-parse', '--verify', '-q', f'refs/heads/{PENDING}{id}', check=False).returncode: return f'nothing is pending as {id}'

def diff(folder, id):
    "What accepting pending candidate `id` would change, as `{ok, id, diff}`."
    if (e := _problem(folder, id)): return dict(ok=False, id=id, error=e)
    return dict(ok=True, id=id, diff=_git(folder, 'diff', f'{PENDING}{id}^', f'{PENDING}{id}').stdout)

def accept(folder, id):
    "Move pending candidate `id` onto the learned folder's HEAD, which the next briefing reads; `{ok, id, commit}`."
    folder = Path(folder)
    if (e := _problem(folder, id)): return dict(ok=False, id=id, error=e)
    with _locked(folder) as held:
        if not held: return dict(ok=False, id=id, error='a Dhrona batch is running on the learned folder; accept it when the batch ends')
        if _git(folder, 'status', '--porcelain').stdout.strip():
            return dict(ok=False, id=id, error=f'{folder} has uncommitted changes; commit or discard them first')
        wt = Path(tempfile.mkdtemp())/'wt'   # picked aside, so a conflict never reaches the files a session briefs from
        try:
            _git(folder, 'worktree', 'add', '-q', '--detach', str(wt), 'HEAD')
            r = _git(wt, *_AS, 'cherry-pick', f'{PENDING}{id}', check=False)
            if r.returncode and 'empty' not in r.stdout + r.stderr:
                return dict(ok=False, id=id, error='it conflicts with what the learned folder holds now; `diff` it, then reject it or edit the folder by hand')
            sha = None if r.returncode else _git(wt, 'rev-parse', 'HEAD').stdout.strip()
            if sha and (n := len((wt/'AGENTS.md').read_text()) if (wt/'AGENTS.md').exists() else 0) > LEARNED_MAX:
                return dict(ok=False, id=id, error=f'accepting it would make AGENTS.md {n:,} characters, past the {LEARNED_MAX:,} Dhrona allows; '
                                                   'reject it, run `/learned compact`, or trim AGENTS.md and commit that first')
            if sha: _git(folder, 'merge', '-q', '--ff-only', sha)
        finally: _git(folder, 'worktree', 'remove', '--force', str(wt), check=False)
        _git(folder, 'branch', '-D', f'{PENDING}{id}')
        return dict(ok=True, id=id, commit=sha[:7] if sha else None)   # None: HEAD already held the change

def compact(folder, chat=None):
    "Ask Dhrona for a shorter `AGENTS.md`, proposed as a pending candidate: `{ok, id, before, after}` in characters, or `{ok: False, error}`."
    try: from dhrona.improve import compact_learned
    except ImportError: return dict(ok=False, error='compacting needs dhrona 0.1.11 or later: uv add "ramabana[dhrona]"')
    r = compact_learned(folder, chat=chat)
    return dict(ok=True, id=r['pending'].removeprefix(PENDING), before=r['before'], after=r['after']) if r['ok'] else r

def learned_command(folder, arg=''):
    "`/learned` and `ramabana-learned` as text: `list` by default, `compact`, or `diff`, `accept` or `reject` with an id."
    cmd, _, id = (arg or 'list').strip().partition(' ')
    if cmd == 'list':
        rows = pending(folder)
        return '\n'.join(f"{p['id']}  {', '.join(p['files'])}  {p['subject']}" for p in rows) or 'nothing is pending'
    if cmd == 'compact':
        r = compact(folder)
        return f"compaction pending as {r['id']}: AGENTS.md {r['before']:,} → {r['after']:,} characters; /learned diff {r['id']} shows it" if r['ok'] else r['error']
    act = dict(diff=diff, accept=accept, reject=reject).get(cmd)
    if act is None: return f'{cmd!r}: say /learned, /learned compact, or /learned diff|accept|reject ID'
    r = act(folder, id.strip())
    if not r['ok']: return r['error']
    if cmd == 'diff': return r['diff'] or '(no change)'
    return f"{cmd}ed {r['id']}" + (f" as {r['commit']}" if r.get('commit') else '')

def reject(folder, id):
    "Drop pending candidate `id`; `{ok, id}`."
    if (e := _problem(folder, id)): return dict(ok=False, id=id, error=e)
    with _locked(folder) as held:
        if not held: return dict(ok=False, id=id, error='a Dhrona batch is running on the learned folder; reject it when the batch ends')
        _git(folder, 'branch', '-D', f'{PENDING}{id}')
        return dict(ok=True, id=id)

A candidate is accepted in a throwaway worktree first, so a conflict never leaves markers in the files a running session briefs from. HEAD moves only when the pick is clean. A candidate whose change HEAD already holds is accepted with `commit` None. The commit keeps Dhrona as its author, and its commit time is the moment of acceptance, which is where Dhrona's 30-day expiry clock starts.

In [ ]:
import os, json, time
def git(d, *a): return subprocess.run(['git', '-C', str(d), *a], check=True, capture_output=True, text=True).stdout.strip()
def propose(folder, files, id):
    "What Dhrona's `keep` does: a commit on `pending/<id>` off HEAD, the folder's files untouched."
    wt = Path(tempfile.mkdtemp())/'wt'
    git(folder, 'worktree', 'add', '-q', '-b', PENDING + id, str(wt), 'HEAD')
    for rel, text in files.items(): (wt/rel).parent.mkdir(parents=True, exist_ok=True); (wt/rel).write_text(text)
    git(wt, 'add', '-A'); git(wt, '-c', 'user.name=Dhrona', '-c', 'user.email=d@x', 'commit', '-qm', f'Lesson from a replay: {id}')
    git(folder, 'worktree', 'remove', '--force', str(wt))

with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'learned'; f.mkdir()
    (f/'AGENTS.md').write_text('A rule a person wrote.\n')
    git(f, 'init', '-q'); git(f, 'add', '-A'); git(f, '-c', 'user.name=t', '-c', 'user.email=t@x', 'commit', '-qm', 'first')
    a, b, c = '20261019-010101-aaaaaa', '20261019-020202-bbbbbb', '20261019-030303-cccccc'
    propose(f, {'skills/sizes/SKILL.md': '---\nname: sizes\ndescription: x\n---\nuse ls\n'}, a)
    propose(f, {'AGENTS.md': 'Something else entirely.\n'}, b)
    propose(f, {'AGENTS.md': 'A third view.\n'}, c)
    test_eq([(p['id'], p['files']) for p in pending(f)], [(a, ['skills/sizes/SKILL.md']), (b, ['AGENTS.md']), (c, ['AGENTS.md'])])
    test_eq((f/'AGENTS.md').read_text(), 'A rule a person wrote.\n')   # nothing pending reaches the briefing
    assert not (f/'skills').exists()
    assert '+use ls' in diff(f, a)['diff']
    r = accept(f, a)
    assert r['ok'] and r['commit'] and (f/'skills'/'sizes'/'SKILL.md').exists() and a not in [p['id'] for p in pending(f)]
    test_eq(git(f, 'log', '-1', '--format=%an'), 'Dhrona')
    test_eq(accept(f, b)['ok'], True)
    head, text = git(f, 'rev-parse', 'HEAD'), (f/'AGENTS.md').read_text()
    r = accept(f, c)                                       # it rewrote the line `b` changed
    assert not r['ok'] and 'conflicts' in r['error'], r
    test_eq((git(f, 'rev-parse', 'HEAD'), (f/'AGENTS.md').read_text()), (head, text))   # HEAD and the files are as they were
    assert '<<<<' not in text and git(f, 'status', '--porcelain') == ''
    test_eq(reject(f, c), dict(ok=True, id=c))
    test_eq(pending(f), [])
    test_eq(accept(f, c)['ok'], False)                     # gone
    test_eq(accept(f, '--force')['ok'], False)             # not an id, never an option
    propose(f, {'AGENTS.md': 'Hand edit pending.\n'}, a)
    (f/'AGENTS.md').write_text('an uncommitted edit\n')
    assert 'uncommitted' in accept(f, a)['error']
    git(f, 'checkout', '-q', '--', '.')
    with _locked(f) as held:                                # a batch holds it
        assert held and 'batch is running' in accept(f, a)['error'] and 'batch is running' in reject(f, a)['error']
    test_eq(len(git(f, 'worktree', 'list').splitlines()), 1)   # no worktree left behind
    test_eq(pending(Path(d)/'none'), [])
    git(f, 'checkout', '-q', '--', '.'); reject(f, a)
    propose(f, {'AGENTS.md': 'A rule a person wrote.\n' + 'x' * LEARNED_MAX}, a)   # past Dhrona's limit
    head, r = git(f, 'rev-parse', 'HEAD'), accept(f, a)
    assert not r['ok'] and 'past the 2,000' in r['error'] and git(f, 'rev-parse', 'HEAD') == head, r
    assert a in learned_command(f) and 'past the 2,000' in learned_command(f, f'accept {a}')
    test_eq(learned_command(f, f'reject {a}'), f'rejected {a}')
    test_eq(learned_command(f), 'nothing is pending')
    assert 'say /learned' in learned_command(f, 'frobnicate')

In [ ]:
with tempfile.TemporaryDirectory() as d:
    f = Path(d)/'learned'; f.mkdir()
    (f/'AGENTS.md').write_text('A rule a person wrote.\n')
    git(f, 'init', '-q'); git(f, 'add', '-A'); git(f, '-c', 'user.name=t', '-c', 'user.email=t@x', 'commit', '-qm', 'first')
    a = '20261019-010101-aaaaaa'
    propose(f, {'AGENTS.md': 'A rule a person wrote.\n\n' + 'Read sizes with ls -l. ' * 40 + '\n'}, a); accept(f, a)
    r = compact(f, chat=lambda prompt: '[kept 1]\n\nRead sizes with ls -l.')
    assert r['ok'] and r['before'] > r['after'] and r['id'] in learned_command(f), r
    test_eq(git(f, 'show', f"{PENDING}{r['id']}:AGENTS.md"), 'A rule a person wrote.\n\nRead sizes with ls -l.')
    assert 'already pending' in learned_command(f, 'compact')
    import sys; mod = sys.modules.pop('dhrona.improve'); sys.modules['dhrona.improve'] = None   # dhrona not installed
    try: assert 'needs dhrona' in compact(f)['error']
    finally: sys.modules['dhrona.improve'] = mod

`learned_command` is the text form a terminal shows. `/learned` in a session and `ramabana-learned` from a shell both use it: with no argument it lists what is pending, `diff ID`, `accept ID` and `reject ID` act on one, and `compact` proposes a shorter `AGENTS.md`.

In [ ]:
#| export
@call_parse
def learned_cli(
    cmd:str='list',               # list, diff, accept, reject or compact
    id:str='',                    # the pending id, for diff, accept and reject
    cfg:str='~/.config/ramabana', # the config folder whose `learned` folder to review
):
    "Review what Dhrona proposed for the learned folder: list it, diff, accept or reject one, or ask for a compaction."
    print(learned_command(learned_folder(cfg), f'{cmd} {id}'.strip()))